<a href="https://colab.research.google.com/github/ewarren38/HW6_WarrenE/blob/main/HW6_WarrenE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Homework 6**
**Author:** Eleanor Warren

**Course:** ST554-601

**Purpose:** Complete HW 6

**Date:** 3.10.26



##**Part 1 | Querying Databases**

First I'm estalblishing a connection to the Lahman baseball database, and then returning a pandas dataframe, `schema_df`, with all the tables in the database.

In [29]:
import pandas as pd
import sqlite3
con = sqlite3.connect("/content/lahman_1871-2022.sqlite")
get_schema = '''
   SELECT *
   FROM sqlite_schema
   WHERE type = 'table';
   '''
schema_df = pd.read_sql(get_schema, con)
schema_df.iloc[6,4]

# The simple querying structure:
# pd.read_sql("SELECT ... FROM ... WHERE...;", con) using con from above.

'CREATE TABLE Batting (\nplayerID TEXT,\nyearID INTEGER,\nstint INTEGER,\nteamID TEXT,\nlgID TEXT,\nG INTEGER,\nG_batting INTEGER,\nAB INTEGER,\nR INTEGER,\nH INTEGER,\n"2B" INTEGER,\n"3B" INTEGER,\nHR INTEGER,\nRBI INTEGER,\nSB INTEGER,\nCS INTEGER,\nBB INTEGER,\nSO INTEGER,\nIBB INTEGER,\nHBP INTEGER,\nSH INTEGER,\nSF INTEGER,\nGIDP INTEGER,\nG_old INTEGER\n)'

Again, since I know I'll be making multiple queries, I'm going to use the in-class example for creating a helper function.

In [14]:
#PROBABLY DELETE

#Make a querying function
def execute_query(connection, query):
    cursor = connection.cursor()
    try:
      cursor.execute(query)
      print("Query executed successfully")
    except Error as e:
      print(f"The error {e} occurred")
    cursor.close()

2. Construct a table of hall of fame pitchers (any hall of famer that pitched) with their `playerID` and sum for `GS, G, W, L, IPOuts, CG, SHO, SV`.

We need to merge together information from the Pitching and the HallOfFame tables.

In [31]:
pitch_stats = pd.read_sql("""
  SELECT DISTINCT p.playerID, sum(p.GS), sum(p.G), sum(p.W), sum(p.L), sum(p.IPOuts), sum(p.CG), sum(p.SHO), sum(p.SV)
  FROM Pitching AS p
  INNER join HallOfFame AS h ON h.playerID = p.playerID
  WHERE inducted = 'Y'
  GROUP BY p.playerID
  ORDER BY p.playerID;
  """, con)

pitch_stats.head()

,playerID,sum(p.GS),sum(p.G),sum(p.W),sum(p.L),sum(p.IPOuts),sum(p.CG),sum(p.SHO),sum(p.SV)
0,alexape01,599,696,373,208,15570,437,90,32
1,ansonca01,0,3,0,1,12,0,0,1
2,becklja01,1,1,0,1,12,0,0,0
3,bendech01,334,459,212,127,9051,255,40,34
4,blylebe01,685,692,287,250,14910,242,60,0


In [32]:
bat_stats = pd.read_sql("""
  SELECT DISTINCT b.playerID, sum(b.AB), sum(b.R), sum(b.H), sum(b.HR), sum(b.RBI), sum(b.BB), sum(b.SO)
  FROM Batting AS b
  INNER join Pitching AS p ON b.playerID = p.playerID
  GROUP BY p.playerID
  ORDER BY p.playerID;
  """, con)

bat_stats.head()

,playerID,sum(b.AB),sum(b.R),sum(b.H),sum(b.HR),sum(b.RBI),sum(b.BB),sum(b.SO)
0,aardsda01,36,0,0,0,0.0,0,18.0
1,aasedo01,65,0,0,0,0.0,0,39.0
2,abadfe01,99,0,11,0,0.0,0,55.0
3,abbeybe01,1350,126,228,0,102.0,126,324.0
4,abbeych01,1756,307,493,19,280.0,167,122.0


In [34]:
HOF_PB_stats = pd.merge(
    left = pitch_stats,
    right = bat_stats,
    how = "inner",
    on = "playerID"
)

HOF_PB_stats

,playerID,sum(p.GS),sum(p.G),sum(p.W),sum(p.L),sum(p.IPOuts),sum(p.CG),sum(p.SHO),sum(p.SV),sum(b.AB),sum(b.R),sum(b.H),sum(b.HR),sum(b.RBI),sum(b.BB),sum(b.SO)
0,alexape01,599,696,373,208,15570,437,90,32,38010,3234,7938,231,3423.0,1617,5796.0
1,ansonca01,0,3,0,1,12,0,0,1,20562,3998,6870,194,4150.0,1968,660.0
2,becklja01,1,1,0,1,12,0,0,0,9551,1603,2938,87,1581.0,616,526.0
3,bendech01,334,459,212,127,9051,255,40,34,18352,1632,3888,96,1856.0,1200,2288.0
4,blylebe01,685,692,287,250,14910,242,60,0,10824,456,1416,0,600.0,120,4632.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,willivi01,471,513,249,205,11988,388,50,11,19409,1391,3224,13,1092.0,1053,2587.0
104,wrighge01,0,3,0,1,15,0,0,0,5746,1330,1732,22,652.0,136,238.0
105,wrighha01,8,36,4,4,301,0,0,14,3252,732,896,16,452.0,148,56.0
106,wynnea01,612,691,300,244,13692,290,49,15,39192,3128,8395,391,3979.0,3243,7590.0
